In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
# Load the dataset
DATA_PATH = "../../data"
CLEAN_DATA_PATH = "../../cleaned_data"
df = pd.read_csv(DATA_PATH + '/train_hai.tsv', sep='\t')
df

,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,Unknown
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,Unknown
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,Unknown
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,Unknown
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,Unknown
...,...,...,...,...,...,...
128172,SDY2867.SUB389678.28___B/Washington/02/2019,SDY2867.SUB389678,28.0,Vic B/Washington/2/2019,20,Unknown
128173,SDY2867.SUB389678.28___H1 HA2 stem,SDY2867.SUB389678,28.0,-,863.42,Unknown
128174,SDY2867.SUB389678.28___H1N1 HA1 (A/Hawaii/70/2...,SDY2867.SUB389678,28.0,H1N1 A/Hawaii/70/2019,42.58,Unknown
128175,SDY2867.SUB389678.7___H1 HA2 stem,SDY2867.SUB389678,7.0,-,874.68,Unknown


In [3]:
df = df.drop(columns=['hai_id', 'material'])
df = df[df['virus_strain'] != '-']
df['value'] = pd.to_numeric(df['value'], errors='coerce')

In [4]:
df['timepoint'].value_counts().sort_index()

timepoint
-7.0         18
 0.0      49017
 3.0        318
 7.0        136
 14.0       777
 28.0     48240
 30.0       468
 70.0       180
 75.0       318
 90.0      2202
 180.0       51
 365.0    23808
Name: count, dtype: int64

In [5]:
# Different strains can skew the average, but day 30 looks close to 28
df.groupby('timepoint')['value'].mean()

timepoint
-7.0       47.777778
 0.0       73.792527
 3.0      160.424528
 7.0      208.851544
 14.0      28.403520
 28.0     142.261548
 30.0     141.481744
 70.0     149.000000
 75.0     317.751572
 90.0     137.563579
 180.0    144.705882
 365.0     83.095474
Name: value, dtype: float64

In [6]:
df['timepoint'] = df['timepoint'].replace(30.0, 28.0)
df = df[df['timepoint'].isin([0.0, 28.0, 365.0])]
df.head()

,participant_id,timepoint,virus_strain,value
0,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.0
1,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.0
2,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.0
3,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.0
4,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.0


In [7]:
df['hai_timepoint'] = 'HAI_' + df['virus_strain'].astype(str) + '_d' + df['timepoint'].astype(int).astype(str)

# Pivot the table
df_pivot = df.pivot_table(
    index='participant_id',
    columns='hai_timepoint',
    values='value'
)

# Reset index to make participant_id a regular column
df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.rename_axis(None, axis=1)

for col in df_pivot.columns:
    if col != 'participant_id':
        df_pivot[col] = df_pivot[col].apply(lambda x: np.log2(x) if pd.notna(x) else x)

df_pivot

,participant_id,HAI_Anc B/Lee/1940_d0,HAI_Anc B/Lee/1940_d28,HAI_Anc B/Lee/1940_d365,HAI_Anc B/Maryland/1959_d0,HAI_Anc B/Maryland/1959_d28,HAI_Anc B/Singapore/1964_d0,HAI_Anc B/Singapore/1964_d28,HAI_H1N1 A/Beijing/262/1995_d0,HAI_H1N1 A/Beijing/262/1995_d28,...,HAI_Yam B/Sichuan/379/1999_d365,HAI_Yam B/Texas/6/2011_d0,HAI_Yam B/Texas/6/2011_d28,HAI_Yam B/Texas/6/2011_d365,HAI_Yam B/Wisconsin/1/2010_d0,HAI_Yam B/Wisconsin/1/2010_d28,HAI_Yam B/Wisconsin/1/2010_d365,HAI_Yam B/Yamagata/16/1988_d0,HAI_Yam B/Yamagata/16/1988_d28,HAI_Yam B/Yamagata/16/1988_d365
0,2016_UGA.ID_001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.321928,5.321928,...,NaN,7.321928,8.321928,NaN,8.321928,8.321928,NaN,7.321928,8.321928,NaN
1,2016_UGA.ID_002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,...,NaN,5.321928,6.321928,NaN,6.321928,6.321928,NaN,5.321928,5.321928,NaN
2,2016_UGA.ID_003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,...,NaN,8.321928,8.321928,NaN,8.321928,8.321928,NaN,8.321928,8.321928,NaN
3,2016_UGA.ID_004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,5.321928,...,NaN,4.321928,6.321928,NaN,5.321928,6.321928,NaN,3.321928,5.321928,NaN
4,2016_UGA.ID_005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,2.321928,...,6.321928,6.321928,6.321928,5.321928,6.321928,6.321928,5.321928,5.321928,5.321928,5.321928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3752,SDY887.SUB134259,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3753,SDY887.SUB134260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3754,SDY887.SUB197783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3755,SDY887.SUB197784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs(CLEAN_DATA_PATH, exist_ok=True)
df_pivot.to_csv(CLEAN_DATA_PATH + '/hai_cleaned.csv', index=False)